In [1]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write("""{
        "file_format_version" : "1.0.0",
        "ICD" : {
            "library_path" : "libEGL_nvidia.so.0"
        }
    }
    """)
        
os.environ["MUJOCO_GL"] = "egl"

import gymnasium
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from PIL import Image
from torchvision.transforms import v2
from einops import rearrange

from config import METAWORLD_CFGS, DMC_CFGS
from metaworld_env import setup_metaworld_env
from dmc import setup_dmc_env

from model.encoder import EncoderNet, FrameObservationEncoderNet
from model.actor import DDPGActorNet
from model.critic import CriticNet

In [2]:
task_name = "pickplace"
seed = 0
feature_layer = -1

In [3]:
def setup_env(task_name:str, seed:int, render_mode:str|None = None) -> gymnasium.Env:
    if task_name in METAWORLD_CFGS:
        env = setup_metaworld_env(task_name, seed, render_mode)
    elif task_name in DMC_CFGS:
        env = setup_dmc_env(task_name, seed, render_mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    env = gymnasium.wrappers.AddRenderObservation(env, render_only=True)
    env = gymnasium.wrappers.FrameStackObservation(env, 2)

    return env

In [ ]:
class SEVEAEncoder(nn.Module):
    def __init__(self, frame_encoder, state_encoder_layer):
        super().__init__()
        self.frame_encoder = frame_encoder
        self.state_encoder_layer = state_encoder_layer

    def forward(self, x):
        x = self.frame_encoder(x)

        if self.state_encoder_layer is not None:
            for layer in self.state_encoder_layer:
                x = layer(x)
                x = F.silu(x)

        return x

In [ ]:
if task_name in METAWORLD_CFGS:
    config_path = "configs/ddpg_metaworld.yaml"
    state_dim = METAWORLD_CFGS[task_name]["state_dim"]
elif task_name in DMC_CFGS:
    config_path = "configs/ddpg_dmc.yaml"
    state_dim = DMC_CFGS[task_name]["state_dim"]

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
envs = setup_env(task_name, 5000, "rgb_array")

obs_dim = envs.observation_space.shape
action_dim = envs.action_space.shape

state_encoder = EncoderNet(state_dim, config["encoder_layers"]).to(device)
actor = DDPGActorNet(state_encoder.dim, np.prod(action_dim), config["actor_layers"]).to(device)
critic = CriticNet(state_encoder.dim, np.prod(action_dim), config["critic_layers"]).to(device)

frame_encoder = FrameObservationEncoderNet(6, state_encoder.dim).to(device)

encoder_weight, actor_weight, critic_weight = torch.load(f"weights/ddpg/{task_name}/actor_best_{seed}.pt", weights_only=True)
state_encoder.load_state_dict(encoder_weight)
actor.load_state_dict(actor_weight)
critic.load_state_dict(critic_weight)

frame_encoder_weight = torch.load(f"weights/ddpg/pickplace/frame_encoder_{seed}_-1.pt", weights_only=True)
frame_encoder.load_state_dict(frame_encoder_weight)



encoder = SEVEAEncoder(frame_encoder, state_encoder.layers[-1:].to(device))
encoder = encoder.eval()
actor = actor.eval()

transform = v2.Compose([
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.485, 0.456, 0.406, 0.485, 0.456, 0.406), (0.229, 0.224, 0.225, 0.229, 0.224, 0.225)),
        ])

In [12]:
def preprpcess(obs_batch):
    obs_batch = rearrange(obs_batch, "b l h w c -> b (l c) h w")
    obs_batch = transform(obs_batch)
    return obs_batch

@torch.no_grad()
def get_action(obs_batch, deterministic, random):

    obs_batch = torch.as_tensor(obs_batch, dtype=torch.float32).unsqueeze(0).to(device)
    #obs_batch = torch.as_tensor(obs_batch).unsqueeze(0).to(device)
    obs_batch = preprpcess(obs_batch)
    obs_batch = encoder(obs_batch)

    dist = actor(obs_batch, 1)
    if deterministic:
        action = dist.mean
    else:    
        action = dist.sample(clip=None)

        if random:
            action.uniform_(-1, 1)
    
    action = action.cpu().numpy()
    
    return action.tolist()

In [13]:
def rollout():
    obs, info = envs.reset()
    success = 0
    for i in range(500):
        action = get_action(obs, deterministic=True, random=False)
        next_obs, reward, done, timeout, info = envs.step(action[0])
        success += info["success"]
        obs = next_obs

    return success

In [14]:
buffer = []
for _ in range(100):
    success = rollout()
    buffer.append(success)

print("Success rate: ", np.mean(buffer))

Success rate:  0.0


In [15]:
buffer

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [16]:
obs, info = envs.reset()
success = 0
imgs = []
for i in range(500):
    imgs.append(Image.fromarray(envs.render()))
    action = get_action(obs, True, False)
    next_obs, reward, done, timeout, info = envs.step(action[0])
    #success += info["success"]
    obs = next_obs

imgs[0].save('output.gif',
             save_all=True,
             append_images=imgs[1:],  # Remaining frames
             duration=100,              # Duration per frame in ms
             loop=0)

In [17]:
info

{'success': 0.0,
 'near_object': 0.0,
 'grasp_success': 0.0,
 'grasp_reward': 0.004624948754207814,
 'in_place_reward': 0.13337174103802216,
 'obj_to_target': 0.3374135788885745,
 'unscaled_reward': 0.004490014053171031,
 'motion': 'unwanted action for push back.',
 'is_grasped': False,
 'episode': {'r': 3.6399389957624524, 'l': 500, 't': 2.156837}}